In [45]:
import numpy as np
from astropy.io import fits
from matplotlib import pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord

## Get Images
```jl
function contains_target(frames, filenames, coords; fk4_coords=nothing, n_cutoff=3, arcsec_threshold=100)

    n_target_frames = 0
    for (i, frame) in enumerate(frames)

        conditions = Dict{String,Any}("NARROWCAM" => frame["CAMNAME"] == "narrow",
                                      "CLEAR GRS" => frame["GRSNAME"] == "clear")
                                      #"OPEN SLIT" => !(haskey(frame, "SLITNAME") && (occursin("vortex", frame["SLITNAME"]) || occursin("corona", frame["SLITNAME"])))

        if !all(values(conditions))
            @warn "Skipping file $(filenames[i]) due to conditions" conditions=conditions
            continue
        end

        if !(frame["RA"] isa Number) || !(frame["DEC"] isa Number)
            @warn "Skipping file $(filenames[i]) due to missing RA/DEC"
            continue
        end

        # need this here or they will get overwritten on subsequent loops
        # for some reason, the coordinate system can vary within an epoch
        ra, dec = coords

        if frame["RADECSYS"] == "FK4"
            @info "Using FK4 coordinates for $(filenames[i])"
            if fk4_coords !== nothing
                ra, dec = fk4_coords
            else
                @warn "Files are in FK4 but no Fk4 coordinates provided!"
            end
        end

        radec_distance = deg2arcsec(small_angle_distance((ra, dec), (frame["RA"], frame["DEC"])))
        @info "Coordinates" object=frame["OBJECT"] target=frame["TARGNAME"] ra=ra dec=dec frame_ra=frame["RA"] frame_dec=frame["DEC"] radec_distance=radec_distance radecsys=frame["RADECSYS"]

        if radec_distance < arcsec_threshold && lowercase(frame["SHRNAME"]) == "open"

            @info "RADEC FRAME" filename=filenames[i] object=frame["OBJECT"] targname=frame["TARGNAME"] radec_distance=radec_distance
            n_target_frames += 1
        end
    end

    if n_target_frames <= n_cutoff
        @warn "Not enough science frames found (only found $(n_target_frames)), skipping obslog generation"
        return false
    end

    return true

end
```

In [ ]:
# in progress
def contains_target(frames, filenames, coords, fk4_coords=None, n_cutoff=3, arcsec_threshold=100):

    n_target_frames = 0

    for i, frame in enumerate(frames):

        conditions = {
            "NARROWCAM" : frame["CAMNAME"] == "narrow",
            "CLEAR GRS" : frame["GRSNAME"] == "clear",
            "OPEN SLIT": not (
                "SLITNAME" in frame and (
                    "vortex" in frame["SLITNAME"] or
                    "corona" in frame["SLITNAME"]
                )
            )
        }

        if np.all(~np.array(list(conditions.values()))):
            print(f"[Warning]: Skipping file {filenames[i]} due to conditions")
            continue

        ## not sure what possible types there are (eg. Number ?)

        # if !(frame["RA"] isa Number) || !(frame["DEC"] isa Number)
        #     @warn "Skipping file $(filenames[i]) due to missing RA/DEC"
        #     continue
        # end

        if frame["RADECSYS"] == "FK4":
            print(f"[Info]: Using FK4 coordinates for {filenames[i]}")
            if fk4_coords != None:
                ra, dec = fk4_coords
            else:
                print("[Warning]: Files are in FK4 but no Fk4 coordinates provided!")

                            #SkyCoord -> c1.separation()
        # radec_distance = (small_angle_distance((ra, dec), (frame["RA"], frame["DEC"])))*3600 # arcseconds per degree
        print(f"[Info]: Coordinates: object={frame["OBJECT"]} target={frame["TARGNAME"]} ra={ra} dec={dec} frame_ra={frame["RA"]} frame_dec={frame["DEC"]} radec_distance={radec_distance} radecsys={frame["RADECSYS"]}")

        # if radec_distance < arcsec_threshold and frame["SHRNAME"].lower() == "open":

        #     print(f"RADEC FRAME filename={filenames[i]} object={frame["OBJECT"]} targname={frame["TARGNAME"]} radec_distance={radec_distance}")
        #     n_target_frames += 1

    if n_target_frames <= n_cutoff:
        print(f"[Warning]: Not enough science frames found (only found {n_target_frames}), skipping obslog generation")
        return False

    return True

In [43]:
A = {1:False,2:False,3:False}
np.all(~np.array(list(A.values())))

np.True_

## Bad Pixel Replacer

```jl
function local_median_replace_bad_pixels!(data, mask, median_size; fail_val=0.0)
    bad_indices = findall(mask)
    half_size = median_size ÷ 2
    height, width = size(data)

    @inbounds for idx in bad_indices
        i, j = Tuple(idx)

        i_start = max(1, i - half_size)
        i_end = min(height, i + half_size)
        j_start = max(1, j - half_size)
        j_end = min(width, j + half_size)

        window = @view data[i_start:i_end, j_start:j_end]
        window_mask = @view mask[i_start:i_end, j_start:j_end]

        # Fast median of good pixels
        good_pixels = window[.!window_mask]
        if length(good_pixels) > 0
            data[i, j] = median(good_pixels)
        else
            data[i, j] = fail_val
        end
    end

end
```

In [ ]:
# done
def median_replace(data:np.array, mask:float, median_size, fail_val:float=0.0):
    '''
    Replaces bad pixels with the median of a box around them.
    '''
    bad_indicies = np.array(np.where(data < mask)).T
    half_size = np.floor(median_size / 2)

    h, w = data.shape
    h_i = h - 1
    w_i = w - 1
    
    for index in bad_indicies:

        i, j = index
        min_i = int(max(0, i-half_size))
        max_i = int(min(i+half_size, w_i) + 1)
        min_j = int(max(0, j-half_size))
        max_j = int(min(j+half_size, h_i) + 1)

        # if no good pixels, use fail_val
        if np.all(data[min_i:max_i, min_j:max_j] < mask):
            data[i,j] = fail_val
        else:
            data[i,j] = np.median(data[min_i:max_i, min_j:max_j])

## Reduce Frame
```
function reduce_frame(frame, master_flats, master_darks, masks; median_size = 6, gain=1.0)

    reduced = copy(frame)

    matched_flat = find_closest_flat(reduced, master_flats)
    matched_dark = find_closest_dark(reduced, master_darks)

    if matched_dark === nothing
        #@warn "No matching dark found for $(reduced["FILENAME"])" reduced["ITIME"] reduced["COADDS"]
        reduced["DARKSUB"] = false
        matched_dark = zeros(size(reduced))
    else
        reduced["DARKSUB"] = true
    end

    if matched_flat === nothing
        #@warn "No matching flat found for $(reduced["FILENAME"])" reduced["FILTER"]
        reduced["FLATDIV"] = false
        matched_flat = ones(size(reduced))
    else
        reduced["FLATDIV"] = true
    end

    reduced = (reduced .- matched_dark) ./ matched_flat

    reduced ./= reduced["COADDS"] # divide by coadds
    reduced .*= gain

    # start with the bad pixel mask as our mask
    mask = copy(NIRC2_bad_pixel_mask)

    if size(mask) != size(reduced)
        mask, _, _ = crop(NIRC2_bad_pixel_mask, size(reduced))
    end

    if haskey(masks, size(reduced))
        mask = mask .| masks[size(reduced)] # also combine the extra mask if it exists
    end

    nan_mask = isnan.(reduced.data) .| isinf.(reduced.data) # create a mask for NaN and Inf values
    mask = mask .| nan_mask # combine the bad pixel mask with the NaN/Inf mask

    local_median_replace_bad_pixels!(reduced.data, mask, median_size)

    reduced_filename = "reduced_$(lpad(reduced["FRAMENO"], 4, '0')).fits"
    reduced["RED-FN"] = reduced_filename

    return reduced

end
```

In [ ]:
## not started
def reduce_frame(frame, master_flats, master_darks, masks, median_size=6, gain=1.0):

    reduced_frame = frame

    return reduced_frame